# E-commerce Customer Segmentation Analysis

This notebook is a concise walkthrough of the reproducible pipeline in `src/segmentation_pipeline.py`. The raw UCI Online Retail workbook is intentionally not committed; see the project README for download and setup instructions.

## 1. Load cleaned transaction data and create RFM features

The pipeline treats completed purchase lines as the analytical base, then aggregates each customer into Recency, Frequency, and Monetary metrics.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from segmentation_pipeline import (
    build_customer_features,
    fit_customer_segments,
    load_and_clean_transactions,
)

transactions, cleaning_audit = load_and_clean_transactions(
    PROJECT_ROOT / 'data/raw/Online Retail.xlsx'
)
customer_features, snapshot_date = build_customer_features(transactions)
print(f'Analysis snapshot: {snapshot_date.date()}')
print(f'Clean purchase lines: {len(transactions):,}')
print(f'Customers: {customer_features.CustomerID.nunique():,}')
customer_features.head()

## 2. Fit four K-Means customer segments

RFM metrics are log-transformed and standardised before K-Means is fitted. The model selection output captures elbow inertia and silhouette scores for transparent review.

In [ ]:
customers, segment_profiles, model_selection = fit_customer_segments(customer_features)
segment_profiles

## 3. Review the committed dashboard outputs

For a fast portfolio review, load the committed, Power BI-ready customer file and profile table directly.

In [ ]:
processed = PROJECT_ROOT / 'data/processed'
dashboard_customers = pd.read_csv(processed / 'customer_segments.csv')
dashboard_profiles = pd.read_csv(processed / 'segment_profiles.csv')
dashboard_profiles[['Segment', 'Customers', 'CustomerSharePct', 'RevenueSharePct', 'AvgRecencyDays', 'RecommendedAction']]

## 4. Business interpretation

The segmentation should guide action rather than merely describe customers. Champions deserve protection and referral activity, Loyal Customers are candidates for cross-sell, Potential Loyalists need a second-purchase nudge, and At Risk customers warrant a targeted win-back sequence. See `reports/POWER_BI_GUIDE.md` for the dashboard model and DAX measures.